# SigLIP 2 Vision-Language Pipeline

This tutorial uses the single pinned `google/siglip2-base-patch16-224` checkpoint packaged by this repository. It demonstrates zero-shot classification, image/text embeddings, cosine similarity, retrieval, and machine-readable provenance.

The tutorial generates deterministic synthetic PPM smoke assets locally; they are not an accuracy benchmark.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kurtvalcorza/siglip2-vision-language-pipeline.git"
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = Path("/content/siglip2-vision-language-pipeline")
    if not ROOT.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    os.chdir(ROOT)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", "requirements.lock.txt"],
        check=True,
    )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-deps",
            "--no-build-isolation",
            "-e",
            ".",
        ],
        check=True,
    )

print(f"repository root: {ROOT.resolve()}")


In [ ]:
import csv
import json
import subprocess
from dataclasses import asdict

import numpy as np

from siglip2_pipeline import load_pipeline, write_provenance

SAMPLE_DIR = ROOT / "outputs" / "sample-data"
subprocess.run(
    [
        sys.executable,
        "examples/sample-data/generate_samples.py",
        "--output-dir",
        str(SAMPLE_DIR),
    ],
    check=True,
)
images = [
    SAMPLE_DIR / "red_square.ppm",
    SAMPLE_DIR / "green_circle.ppm",
    SAMPLE_DIR / "blue_triangle.ppm",
]
labels = ["red square", "green circle", "blue triangle", "abstract geometric shape"]
print("samples:", [path.name for path in images])


In [ ]:
pipe = load_pipeline()
print("loaded pinned SigLIP 2 checkpoint")


In [ ]:
classification = pipe.zero_shot_classify(images[0], labels)
for item in classification:
    print(f"{item.label:28s} {item.score:.6f}")

# Scores are independent SigLIP sigmoids; they are not calibrated probabilities
# and are not expected to sum to one.


In [ ]:
image_embeddings = pipe.embed_image(images)
text_embeddings = pipe.embed_text(labels[:3])
similarity = pipe.similarity(images, labels[:3])

print("image embeddings:", image_embeddings.shape)
print("text embeddings:", text_embeddings.shape)
print("similarity matrix:")
print(np.array2string(similarity, precision=4))


In [ ]:
retrieval = pipe.retrieve("a blue triangle", images, top_k=3)
for hit in retrieval:
    print(images[hit.index].name, f"{hit.score:.6f}")


In [ ]:
OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(exist_ok=True)

(OUTPUT / "classification.json").write_text(
    json.dumps([asdict(item) for item in classification], indent=2) + "\n",
    encoding="utf-8",
)
(OUTPUT / "retrieval.json").write_text(
    json.dumps(
        [
            {"index": hit.index, "filename": images[hit.index].name, "score": hit.score}
            for hit in retrieval
        ],
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)

with (OUTPUT / "similarity.csv").open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["image", *labels[:3]])
    for image_path, row in zip(images, similarity, strict=True):
        writer.writerow([image_path.name, *[float(value) for value in row]])

write_provenance(OUTPUT / "provenance.json")
print("wrote:", sorted(path.name for path in OUTPUT.iterdir()))


## BYOD

Replace any path in `images` with a local image path after upload. Remote HTTP(S) image strings are rejected by the pipeline by design. Candidate labels and prompts are part of the inference configuration and should be validated for the intended domain.

This tutorial does not provide object detection, segmentation, OCR, caption generation, calibrated probabilities, or a universal decision threshold.
